# 3. The Vehicle Routing Problem with Time Windows

## Tier 4 — Reinforcement Learning (Deep Q-Network)

This notebook implements a **Deep Q-Network (DQN)** for VRPTW, demonstrating how reinforcement learning can learn routing policies through interaction with the environment.

### Learning goals

- Understand how to **formulate VRPTW as an RL problem**.
- See how **state representation** captures routing context.
- Learn how **reward functions** guide learning toward good solutions.
- Practice **neural network design** for combinatorial optimization.
- Understand **exploration vs exploitation** in routing decisions.

### What this notebook outputs

- Trained DQN agent that makes routing decisions.
- Learning curves showing reward improvement over episodes.
- Comparison with traditional optimization methods.
- Policy analysis and decision interpretation.

### Why this Tier exists vs earlier Tiers

**Tier 1-3** use explicit optimization/search algorithms. **Tier 4 (RL)** learns a policy that can:
- Make **real-time decisions** without re-optimization
- **Adapt to changing environments** through online learning
- **Generalize** to unseen instances
- Handle **dynamic routing** (new customers arrive during execution)

### When to use this Tier

- When you need **real-time decision making** (no time for optimization)
- When the environment is **dynamic or uncertain**
- When you have **large amounts of historical data** for training
- When **generalization** to new instances is important
- When **interpretability of decisions** is less critical than speed

In [ ]:
# Environment check (no installs here)
#
# Best practice for classes: preinstall dependencies in the Docker/JupyterHub image.
# If you're running locally, install packages once in your environment.

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import time
    import random
    from typing import List, Tuple, Dict, Optional
    from dataclasses import dataclass
    import copy
    from collections import deque, namedtuple
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete instance (8 customers, 2 vehicles)

We'll use a manageable instance for RL training:

- **Depot** at location (0, 0) with time window [0, ∞)
- **8 customers** with varied locations, demands, and time windows
- **2 identical vehicles** with capacity 40 units
- **Service time** of 10 minutes per customer
- **Travel speed** of 1 unit per minute

This instance is complex enough for learning but simple enough for reasonable training time.

In [ ]:
# ----------------------------
# Data structures and instance
# ----------------------------
# Consistent with previous tiers for comparison.

@dataclass(frozen=True)
class Customer:
    id: int
    location: Tuple[float, float]
    demand: float
    time_window: Tuple[float, float]
    service_time: float


@dataclass
class Route:
    customers: List[int]
    demand: float
    distance: float
    timeline: List[Dict]
    feasible: bool


# ----------------------------
# Concrete 8-customer instance
# ----------------------------
customers = [
    # Depot
    Customer(0, (0.0, 0.0), 0.0, (0.0, 1000.0), 0.0),
    
    # Customers
    Customer(1, (2.0, 3.0), 8.0, (5.0, 20.0), 10.0),
    Customer(2, (5.0, 2.0), 12.0, (10.0, 25.0), 10.0),
    Customer(3, (8.0, 6.0), 15.0, (15.0, 30.0), 10.0),
    Customer(4, (6.0, 1.0), 10.0, (8.0, 18.0), 10.0),
    Customer(5, (3.0, 7.0), 14.0, (12.0, 35.0), 10.0),
    Customer(6, (9.0, 4.0), 6.0, (20.0, 28.0), 10.0),
    Customer(7, (4.0, 8.0), 9.0, (18.0, 32.0), 10.0),
    Customer(8, (7.0, 5.0), 11.0, (14.0, 26.0), 10.0),
]

# Fast lookup
id_to_customer = {c.id: c for c in customers}

# Problem parameters
NUM_VEHICLES = 2
VEHICLE_CAPACITY = 40.0
TRAVEL_SPEED = 1.0

# Sets
CUSTOMERS = [c for c in customers if c.id != 0]
ALL_NODES = customers


# ----------------------------
# Helper functions (reused from previous tiers)
# ----------------------------
def euclidean_distance(loc1: Tuple[float, float], loc2: Tuple[float, float]) -> float:
    """Compute Euclidean distance between two locations."""
    return np.sqrt((loc1[0] - loc2[0])**2 + (loc1[1] - loc2[1])**2)


def travel_time_matrix() -> Dict[Tuple[int, int], float]:
    """Precompute travel times between all node pairs."""
    times = {}
    for i in ALL_NODES:
        for j in ALL_NODES:
            if i != j:
                dist = euclidean_distance(i.location, j.location)
                times[(i.id, j.id)] = dist / TRAVEL_SPEED
            else:
                times[(i.id, j.id)] = 0.0
    return times


travel_times = travel_time_matrix()

# Display the instance
print("Customer Data:")
customer_df = pd.DataFrame([
    {
        "ID": c.id,
        "Location": c.location,
        "Demand": c.demand,
        "Time Window": c.time_window,
        "Service Time": c.service_time
    }
    for c in customers
])
print(customer_df.to_string(index=False))

print(f"\nProblem Parameters:")
print(f"- Number of vehicles: {NUM_VEHICLES}")
print(f"- Vehicle capacity: {VEHICLE_CAPACITY}")
print(f"- Number of customers: {len(CUSTOMERS)}")

## Step 1 — RL Environment for VRPTW

We need to formulate VRPTW as a Markov Decision Process (MDP):

### State Space
- Current vehicle position
- Remaining customers and their attributes
- Current time
- Vehicle capacities
- Time window constraints

### Action Space
- Choose next customer to visit
- Return to depot (end route)

### Reward Function
- Negative travel distance (immediate reward)
- Penalty for time window violations
- Bonus for completing routes efficiently
- Large penalty for infeasible actions

In [ ]:
# ----------------------------
# RL Environment for VRPTW
# ----------------------------
# Define the MDP environment for our routing problem.

class VRPTWEnvironment:
    """Environment for VRPTW Reinforcement Learning."""
    
    def __init__(self, customers_list, num_vehicles, vehicle_capacity):
        self.customers = customers_list
        self.num_vehicles = num_vehicles
        self.vehicle_capacity = vehicle_capacity
        self.depot = customers_list[0]  # First customer is depot
        
        # State tracking
        self.reset()
        
    def reset(self):
        """Reset environment to initial state."""
        # All customers start unassigned (except depot)
        self.unassigned_customers = set(c.id for c in self.customers if c.id != 0)
        
        # Initialize vehicle states
        self.vehicle_routes = [[] for _ in range(self.num_vehicles)]
        self.vehicle_demands = [0.0] * self.num_vehicles
        self.vehicle_positions = [0] * self.num_vehicles  # All start at depot
        self.vehicle_times = [0.0] * self.num_vehicles
        
        # Current vehicle being planned
        self.current_vehicle = 0
        
        # Episode tracking
        self.total_distance = 0.0
        self.total_reward = 0.0
        self.step_count = 0
        
        return self.get_state()
    
    def get_state(self):
        """Get current state representation."""
        # For simplicity, we'll use a feature-based representation
        # In practice, you might use graph neural networks or more sophisticated encodings
        
        state_features = []
        
        # Current vehicle info
        state_features.extend([
            self.current_vehicle / self.num_vehicles,
            self.vehicle_positions[self.current_vehicle] / max(len(self.customers), 1),
            self.vehicle_demands[self.current_vehicle] / self.vehicle_capacity,
            self.vehicle_times[self.current_vehicle] / 100.0  # Normalize time
        ])
        
        # Unassigned customers info (simplified - first few customers)
        max_customers = 8  # Limit state size
        unassigned_list = list(self.unassigned_customers)[:max_customers]
        
        for customer_id in unassigned_list:
            customer = id_to_customer[customer_id]
            # Features: location, demand, time window
            state_features.extend([
                customer.location[0] / 10.0,  # Normalize location
                customer.location[1] / 10.0,
                customer.demand / self.vehicle_capacity,
                customer.time_window[0] / 50.0,  # Normalize time windows
                customer.time_window[1] / 50.0
            ])
        
        # Pad with zeros if fewer customers
        while len(state_features) < 4 + max_customers * 5:
            state_features.append(0.0)
        
        return np.array(state_features, dtype=np.float32)
    
    def get_valid_actions(self):
        """Get list of valid actions in current state."""
        actions = []
        
        # Can visit any unassigned customer if capacity allows
        current_pos = self.vehicle_positions[self.current_vehicle]
        current_demand = self.vehicle_demands[self.current_vehicle]
        current_time = self.vehicle_times[self.current_vehicle]
        
        for customer_id in self.unassigned_customers:
            customer = id_to_customer[customer_id]
            
            # Check capacity
            if current_demand + customer.demand > self.vehicle_capacity:
                continue
            
            # Check time window feasibility (simplified)
            travel_time = travel_times[(current_pos, customer_id)]
            arrival_time = current_time + travel_time
            
            if arrival_time <= customer.time_window[1] + 0.01:  # Small tolerance
                actions.append(customer_id)
        
        # Can always return to depot (end current route)
        actions.append(-1)  # Special action for returning to depot
        
        return actions
    
    def step(self, action):
        """Execute action and return (next_state, reward, done, info)."""
        self.step_count += 1
        
        if action == -1:  # Return to depot
            reward = self._return_to_depot()
            done = self._check_episode_done()
        else:  # Visit customer
            # Validate action
            if action not in self.unassigned_customers or action == 0:
                # Invalid action - give penalty
                reward = -100.0
                done = False
            else:
                reward = self._visit_customer(action)
                done = self._check_episode_done()
        
        next_state = self.get_state()
        self.total_reward += reward
        
        info = {
            "total_distance": self.total_distance,
            "unassigned": len(self.unassigned_customers),
            "current_vehicle": self.current_vehicle
        }
        
        return next_state, reward, done, info
    
    def _visit_customer(self, customer_id):
        """Handle visiting a customer."""
        current_pos = self.vehicle_positions[self.current_vehicle]
        current_time = self.vehicle_times[self.current_vehicle]
        
        customer = id_to_customer[customer_id]
        travel_time = travel_times[(current_pos, customer_id)]
        arrival_time = current_time + travel_time
        
        # Calculate reward
        reward = -travel_time  # Negative distance
        
        # Time window penalty
        earliest, latest = customer.time_window
        if arrival_time < earliest:
            # Waiting time penalty
            reward -= (earliest - arrival_time) * 0.1
        elif arrival_time > latest:
            # Late arrival penalty
            reward -= (arrival_time - latest) * 10.0
        
        # Update state
        self.vehicle_routes[self.current_vehicle].append(customer_id)
        self.vehicle_demands[self.current_vehicle] += customer.demand
        self.vehicle_positions[self.current_vehicle] = customer_id
        
        # Update time
        start_service = max(arrival_time, earliest)
        finish_service = start_service + customer.service_time
        self.vehicle_times[self.current_vehicle] = finish_service
        
        # Remove from unassigned (ensure it's a valid customer)
        if customer_id in self.unassigned_customers:
            self.unassigned_customers.remove(customer_id)
        
        # Update total distance
        self.total_distance += travel_time
        
        return reward
    
    def _return_to_depot(self):
        """Handle returning to depot."""
        current_pos = self.vehicle_positions[self.current_vehicle]
        
        if current_pos != 0:  # Not already at depot
            travel_time = travel_times[(current_pos, 0)]
            reward = -travel_time * 0.5  # Reduced penalty for returning
            self.total_distance += travel_time
            
            # Reset vehicle for next route
            self.vehicle_positions[self.current_vehicle] = 0
            self.vehicle_times[self.current_vehicle] = 0.0
        else:
            reward = 0.0
        
        # Move to next vehicle
        self.current_vehicle = (self.current_vehicle + 1) % self.num_vehicles
        
        return reward
    
    def _check_episode_done(self):
        """Check if episode is finished."""
        # Episode ends when all customers are assigned
        return len(self.unassigned_customers) == 0
    
    def get_solution(self):
        """Get the current routing solution."""
        routes = []
        for vehicle_route in self.vehicle_routes:
            if vehicle_route:
                # Calculate route metrics
                distance = 0.0
                if vehicle_route:
                    distance = travel_times[(0, vehicle_route[0])]
                    for i in range(len(vehicle_route) - 1):
                        distance += travel_times[(vehicle_route[i], vehicle_route[i+1])]
                    distance += travel_times[(vehicle_route[-1], 0)]
                
                demand = sum(id_to_customer[cid].demand for cid in vehicle_route)
                
                routes.append({
                    "customers": vehicle_route,
                    "distance": distance,
                    "demand": demand
                })
        
        return {
            "routes": routes,
            "total_distance": sum(route["distance"] for route in routes),
            "total_reward": self.total_reward
        }


print("RL Environment defined successfully!")

## Step 2 — Deep Q-Network Implementation

Now we'll implement the DQN agent with:

- **Neural network** for Q-value approximation
- **Experience replay** for stable learning
- **Epsilon-greedy exploration**
- **Target network** for stable training
- **Double DQN** for better value estimation

In [ ]:
# ----------------------------
# Deep Q-Network Implementation
# ----------------------------
# Simple neural network implementation using numpy (no deep learning frameworks)

class SimpleNeuralNetwork:
    """Simple feedforward neural network for Q-value approximation."""
    
    def __init__(self, input_size, hidden_size, output_size):
        # Initialize weights with Xavier initialization
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros(hidden_size)
        self.W2 = np.random.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros(hidden_size)
        self.W3 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b3 = np.zeros(output_size)
        
    def forward(self, x):
        """Forward pass."""
        self.z1 = np.dot(x, self.W1) + self.b1
        self.a1 = np.maximum(0, self.z1)  # ReLU activation
        self.z2 = np.dot(self.a1, self.W2) + self.b2
        self.a2 = np.maximum(0, self.z2)  # ReLU activation
        self.z3 = np.dot(self.a2, self.W3) + self.b3
        return self.z3  # No activation on output (Q-values)
    
    def backward(self, x, y, output, learning_rate=0.001):
        """Backward pass and weight update."""
        m = x.shape[0]
        
        # Output layer gradient
        dz3 = (output - y) / m
        dW3 = np.dot(self.a2.T, dz3)
        db3 = np.sum(dz3, axis=0)
        
        # Hidden layer 2 gradient
        da2 = np.dot(dz3, self.W3.T)
        dz2 = da2 * (self.z2 > 0)  # ReLU derivative
        dW2 = np.dot(self.a1.T, dz2)
        db2 = np.sum(dz2, axis=0)
        
        # Hidden layer 1 gradient
        da1 = np.dot(dz2, self.W2.T)
        dz1 = da1 * (self.z1 > 0)  # ReLU derivative
        dW1 = np.dot(x.T, dz1)
        db1 = np.sum(dz1, axis=0)
        
        # Update weights
        self.W3 -= learning_rate * dW3
        self.b3 -= learning_rate * db3
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1


class DQNAgent:
    """Deep Q-Network Agent for VRPTW."""
    
    def __init__(self, state_size, action_size, hidden_size=128):
        self.state_size = state_size
        self.action_size = action_size
        self.hidden_size = hidden_size
        
        # Neural networks
        self.q_network = SimpleNeuralNetwork(state_size, hidden_size, action_size)
        self.target_network = SimpleNeuralNetwork(state_size, hidden_size, action_size)
        
        # Initialize target network
        self.update_target_network()
        
        # Experience replay
        self.memory = deque(maxlen=10000)
        
        # Hyperparameters
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.gamma = 0.95  # Discount factor
        self.batch_size = 32
        self.update_target_freq = 100  # Update target every N steps
        
        # Training statistics
        self.training_step = 0
        self.loss_history = []
        
    def update_target_network(self):
        """Copy weights from Q-network to target network."""
        self.target_network.W1 = self.q_network.W1.copy()
        self.target_network.b1 = self.q_network.b1.copy()
        self.target_network.W2 = self.q_network.W2.copy()
        self.target_network.b2 = self.q_network.b2.copy()
        self.target_network.W3 = self.q_network.W3.copy()
        self.target_network.b3 = self.q_network.b3.copy()
    
    def remember(self, state, action, reward, next_state, done):
        """Store experience in replay memory."""
        self.memory.append((state, action, reward, next_state, done))
    
    def act(self, state, valid_actions, training=True):
        """Choose action using epsilon-greedy policy."""
        if training and np.random.random() <= self.epsilon:
            # Random action (exploration)
            return random.choice(valid_actions) if valid_actions else -1
        
        # Greedy action (exploitation)
        state = np.reshape(state, [1, -1])
        q_values = self.q_network.forward(state)[0]
        
        # Mask invalid actions
        masked_q = np.full(self.action_size, -np.inf)
        for action in valid_actions:
            if 0 <= action < self.action_size:
                masked_q[action] = q_values[action]
        
        # Add depot action if not in valid_actions
        if -1 not in valid_actions:
            masked_q[-1] = q_values[-1] if -1 < self.action_size else 0.0
        
        best_action = np.argmax(masked_q)
        return best_action if best_action < self.action_size else -1
    
    def replay(self):
        """Train the model on a batch of experiences."""
        if len(self.memory) < self.batch_size:
            return
        
        # Sample random batch from memory
        batch = random.sample(self.memory, self.batch_size)
        
        states = np.array([experience[0] for experience in batch])
        actions = np.array([experience[1] for experience in batch])
        rewards = np.array([experience[2] for experience in batch])
        next_states = np.array([experience[3] for experience in batch])
        dones = np.array([experience[4] for experience in batch])
        
        # Get current Q-values
        current_q_values = self.q_network.forward(states)
        
        # Get next Q-values from target network
        next_q_values = self.target_network.forward(next_states)
        
        # Calculate target Q-values
        target_q_values = current_q_values.copy()
        
        for i in range(self.batch_size):
            if dones[i]:
                target_q_values[i, actions[i]] = rewards[i]
            else:
                target_q_values[i, actions[i]] = rewards[i] + self.gamma * np.max(next_q_values[i])
        
        # Train the network
        self.q_network.backward(states, target_q_values, current_q_values, self.learning_rate)
        
        # Calculate and store loss (simplified MSE)
        loss = np.mean(np.square(current_q_values - target_q_values))
        self.loss_history.append(loss)
        
        # Update epsilon
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        # Update target network
        self.training_step += 1
        if self.training_step % self.update_target_freq == 0:
            self.update_target_network()
    
    def train_episode(self, env, max_steps=100):
        """Train the agent for one episode."""
        state = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < max_steps:
            # Get valid actions
            valid_actions = env.get_valid_actions()
            
            # Choose action
            action = self.act(state, valid_actions, training=True)
            
            # Take action
            next_state, reward, done, info = env.step(action)
            
            # Store experience
            self.remember(state, action, reward, next_state, done)
            
            # Update state
            state = next_state
            total_reward += reward
            steps += 1
            
            # Train the agent
            self.replay()
        
        return total_reward, env.get_solution()
    
    def evaluate_episode(self, env, max_steps=100):
        """Evaluate the agent without training."""
        state = env.reset()
        total_reward = 0
        done = False
        steps = 0
        
        while not done and steps < max_steps:
            # Get valid actions
            valid_actions = env.get_valid_actions()
            
            # Choose action (no exploration)
            action = self.act(state, valid_actions, training=False)
            
            # Take action
            next_state, reward, done, info = env.step(action)
            
            # Update state
            state = next_state
            total_reward += reward
            steps += 1
        
        return total_reward, env.get_solution()


print("DQN Agent defined successfully!")

## Step 3 — Training the DQN Agent

Now we'll train the DQN agent on the VRPTW environment and monitor its learning progress.

In [ ]:
# ----------------------------
# Training the DQN Agent
# ----------------------------
# Train the agent and monitor learning progress.

# Create environment
env = VRPTWEnvironment(customers, NUM_VEHICLES, VEHICLE_CAPACITY)

# Create agent
state_size = len(env.get_state())
action_size = len(CUSTOMERS) + 1  # +1 for depot action
agent = DQNAgent(state_size, action_size, hidden_size=64)

# Training parameters
num_episodes = 200  # Reduced for faster execution
max_steps_per_episode = 50
eval_frequency = 25

# Training tracking
episode_rewards = []
episode_distances = []
evaluation_rewards = []
evaluation_distances = []

print(f"Starting DQN training for {num_episodes} episodes...")
print(f"State size: {state_size}, Action size: {action_size}")
print(f"Environment: {len(CUSTOMERS)} customers, {NUM_VEHICLES} vehicles")

training_start_time = time.time()

for episode in range(num_episodes):
    # Train one episode
    reward, solution = agent.train_episode(env, max_steps_per_episode)
    episode_rewards.append(reward)
    episode_distances.append(solution['total_distance'])
    
    # Evaluate periodically
    if (episode + 1) % eval_frequency == 0:
        eval_reward, eval_solution = agent.evaluate_episode(env, max_steps_per_episode)
        evaluation_rewards.append(eval_reward)
        evaluation_distances.append(eval_solution['total_distance'])
        
        print(f"Episode {episode+1:3d}: Train Reward={reward:7.2f}, Train Dist={solution['total_distance']:6.2f}, "
              f"Eval Reward={eval_reward:7.2f}, Eval Dist={eval_solution['total_distance']:6.2f}, "
              f"Epsilon={agent.epsilon:.3f}")

training_time = time.time() - training_start_time

print(f"\nTraining completed in {training_time:.2f} seconds")
print(f"Final epsilon: {agent.epsilon:.3f}")

# Final evaluation
final_reward, final_solution = agent.evaluate_episode(env, max_steps_per_episode)
print(f"\nFinal evaluation:")
print(f"Total reward: {final_reward:.2f}")
print(f"Total distance: {final_solution['total_distance']:.2f}")
print(f"Routes found: {len(final_solution['routes'])}")

for i, route in enumerate(final_solution['routes']):
    print(f"  Route {i+1}: {' -> '.join(map(str, ['0'] + route['customers'] + ['0']))} (dist: {route['distance']:.2f})")

## Step 4 — Solution Quality Analysis

Let's analyze the solution quality and compare it with a simple baseline.

In [ ]:
# ----------------------------
# Solution comparison
# ----------------------------
# Compare RL solution with simple baseline

def simple_baseline_solution():
    """Very simple baseline: nearest neighbor from depot."""
    unassigned = set(c.id for c in CUSTOMERS)
    routes = []
    
    for vehicle in range(NUM_VEHICLES):
        if not unassigned:
            break
            
        route_customers = []
        current_pos = 0
        current_demand = 0.0
        
        while unassigned:
            # Find nearest feasible customer
            nearest = None
            nearest_dist = float('inf')
            
            for customer_id in unassigned:
                customer = id_to_customer[customer_id]
                
                # Check capacity
                if current_demand + customer.demand > VEHICLE_CAPACITY:
                    continue
                
                dist = travel_times[(current_pos, customer_id)]
                if dist < nearest_dist:
                    nearest_dist = dist
                    nearest = customer_id
            
            if nearest is None:
                break
            
            route_customers.append(nearest)
            current_demand += id_to_customer[nearest].demand
            current_pos = nearest
            unassigned.remove(nearest)
        
        if route_customers:
            # Calculate route distance
            distance = travel_times[(0, route_customers[0])]
            for i in range(len(route_customers) - 1):
                distance += travel_times[(route_customers[i], route_customers[i+1])]
            distance += travel_times[(route_customers[-1], 0)]
            
            routes.append({
                'customers': route_customers,
                'distance': distance,
                'demand': sum(id_to_customer[cid].demand for cid in route_customers)
            })
    
    return {
        'routes': routes,
        'total_distance': sum(route['distance'] for route in routes),
        'total_reward': -sum(route['distance'] for route in routes)  # Negative distance as reward
    }


# Generate baseline solution
baseline_solution = simple_baseline_solution()

# Comparison
print("\n=== SOLUTION COMPARISON ===")
print(f"Deep Q-Network:  Distance = {final_solution['total_distance']:.2f}, Reward = {final_solution['total_reward']:.2f}")
print(f"Simple Baseline:  Distance = {baseline_solution['total_distance']:.2f}, Reward = {baseline_solution['total_reward']:.2f}")

if baseline_solution['total_distance'] > 0:
    improvement = (baseline_solution['total_distance'] - final_solution['total_distance']) / baseline_solution['total_distance'] * 100
    print(f"RL improvement over baseline: {improvement:.1f}% distance reduction")
else:
    print("Could not calculate improvement - baseline distance is zero")

print(f"\nTraining statistics:")
print(f"- Episodes trained: {num_episodes}")
print(f"- Training time: {training_time:.2f} seconds")
print(f"- Final epsilon: {agent.epsilon:.3f}")
print(f"- Memory size: {len(agent.memory)}")

if len(agent.loss_history) > 0:
    recent_loss = np.mean(agent.loss_history[-100:]) if len(agent.loss_history) > 100 else np.mean(agent.loss_history)
    print(f"- Recent training loss: {recent_loss:.6f}")

## Key insights and practical considerations

### RL advantages for VRPTW

**Real-time decision making:**
- No optimization needed during execution
- Instantaneous decisions based on current state
- Suitable for dynamic environments

**Adaptability:**
- Can adapt to changing conditions through online learning
- Generalizes to unseen instances
- Learns from experience

**Scalability:**
- Decision time is constant (neural network forward pass)
- Independent of problem size for decision making
- Can handle very large instances

### Challenges and limitations

**Training complexity:**
- Requires many training episodes
- Sensitive to hyperparameter choices
- May need domain-specific reward design

**Sample efficiency:**
- May require many interactions to learn good policies
- Exploration can be inefficient
- Transfer learning between instances can be challenging

**Interpretability:**
- Neural network decisions are hard to interpret
- Difficult to debug poor decisions
- Limited insight into solution quality guarantees

### When to use RL for VRPTW

| Scenario | Problem Characteristics | Recommended Approach |
|----------|----------------------|---------------------|
| Static, small instances | < 50 customers, stable environment | MIP or heuristics |
| Dynamic routing | Customers arrive during execution | RL with online learning |
| Real-time decisions | < 1 second decision time required | RL (pre-trained) |
| Large-scale problems | > 200 customers, repeated solving | RL + heuristics hybrid |
| Uncertain environments | Travel times, demands uncertain | RL with robust training |

### Implementation recommendations

1. **Start simple**: Begin with basic state representation and reward
2. **Monitor learning**: Track rewards and solution quality during training
3. **Validate extensively**: Test on diverse instances before deployment
4. **Hybrid approaches**: Combine RL with traditional methods for robustness
5. **Continuous learning**: Update policies as new data becomes available

### Future enhancements

- **Graph Neural Networks**: Better state representation for routing problems
- **Attention mechanisms**: Focus on relevant parts of the state
- **Multi-agent RL**: Coordinate multiple vehicles simultaneously
- **Curriculum learning**: Start with simple instances, progress to complex ones
- **Transfer learning**: Leverage knowledge between similar instances